# Домашнее задание №4

Материалом для практики послужит текст из [книги](https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615), содержащей переписку писателя-драматурга Николая Эрдмана и его возлюбленной Ангелиной Степановой, актрисы МХАТа. 
В тексте есть сами письма (с 1928 по 1935 гг.), а также фрагменты из воспоминаний Степановой (опубликованных в 1995 году и относящихся к этому периоду).


Ваша задача — подготовить датасет писем для дальнейшего количественного анализа.

Необходимо сделать pandas датафрейм, в котором буду храниться следующие сведения (столбцы): 
- дата написания письма (желательно в формате YYYY-MM-DD), 
- автор (Степанова или Эрдман), 
- текст письма, 
- лемматизированный текст письма (без знаков препинания), 
- количество слов в письме. 

> Воспоминания сохранять в датафрейм НЕ НУЖНО!
> Сноски вида [23] и переносы строк нужно удалить из текста.

Чтобы извлечь письма и метаданные о них, нужно воспользоваться [регулярными выражениями](https://docs.python.org/3/library/re.html). 

Обратите внимание, что даты оформлены не единообразно. Продумайте все возможные варианты, перепроверьте данные. Адресант письма так же может быть указан не единообразно или не указан вовсе (но, например, есть адресат). Задача — извлечь метаданные для наибольшего количества писем.

После окончания работы над датасетом необходимо вывести df.info(), чтобы можно было посмотреть количество извлечённых дат, а также valuecounts() для столбца с адресантами.

### КРИТЕРИИ

| Что сделано | Баллы |
|---|---:|
| Извлечены тексты писем, собраны в датафрейм с указанными столбцами | 2 |
| Текст очищен и лемматизирован | 1 |
| Автор каждого письма определён: «Степанова» или «Эрдман» (включая случаи, когда автор указан не единообразно или восстанавливается через адресата) | 2 |
| Все имеющиеся даты приведены к единому формату (в оригинале даты не единообразны, например, «1928», «января 1928», «12.02.1928», «12 февраля 1928») | 3 |
| Датасет опубликован на GitHub в отдельном репозитории с файлом README.md (описание источника, структуры столбцов, методов извлечения и очистки, описаны принципы работы с отсутствующими значениями дат). Достаточно прикрепить ссылку на репозиторий. Датасет нужно опубликовать в формате .csv. | 2 |

Для поднятия настроения — басня Эрдмана:

```
СМЕТАНА.

Мы любим подмечать у недругов изъяны,
И направлять на них насмешки остриё.

Однажды молоко спросило у сметаны:
- Скажите, вы еда или питье?
Сметана молвила:
- Оставьте ваши шутки!
Действительно, я где-то в промежутке,
Но ведь важна не эта сторона
Всего важнее то, что я вкусна
И все, как правило, бывают мною сыты.

Вот так же точно и гермафродиты:
Тот, кто на свет их произвел,
Конечно, допустил ужасную небрежность.
Но ведь в конце-концов
существенен не пол,
А классовая принадлежность.
```

Читать другие басни: https://stihi.ru/diary/valery31/2014-10-11

# Решение ДЗ | Прищепа Богдан

In [1]:
%pip install requests pandas pymorphy3 razdel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Загрузка текста

In [2]:
import os
import re
import requests
import pandas as pd
import pymorphy3
from razdel import tokenize

In [5]:
SOURCE_URL = "https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615"
MIRROR_URL = "https://moreknig.org/reader/28736/"
TEXT_URL = "https://d.moreknig.org/engine/download.php?id=86199"

print("рабочая папка:", os.getcwd())

if not os.path.exists("book.txt"):
    response = requests.get(TEXT_URL, timeout=60)
    response.raise_for_status()
    book = response.content.decode("cp1251")

    if "Воспоминания" not in book or "Степановой" not in book:
        raise ValueError("вместо текста книги скачалась другая страница")

    with open("book.txt", "w", encoding="utf-8") as file:
        file.write(book)

with open("book.txt", "r", encoding="utf-8") as file:
    book = file.read()

print("кол-во символов в книге:", len(book))

рабочая папка: c:\Users\Admin\Desktop\Данные для Python
кол-во символов в книге: 111569


## Работа с текстом -- отделяю письма от воспоминаний

In [ ]:
text = book.replace("\xa0", " ").replace("\r", "")

text = re.split(r"(?m)^\s*В\.\s*Вульф\. Вместо послесловия", text)[0]
text = re.split(r"(?m)^Они встретились в квартире Бориса Робертовича", text)[0]

note_part = re.search(
    r"Записки Н\.\s*Эрдмана.*?1931 года\.(.*?)(?=\n\s*\*\s*\*\s*\*)",
    text, re.S
)
notes = []
if note_part:
    notes = re.findall(r"«(.*?)»", note_part[1], re.S)



text = re.sub(
    r"(?m)^(Предчувствие меня не обмануло, свидание состоялось)",
    r"* * *\n\1", text
)

text = re.sub(
    r"\[воспоминания[^\]]*\].*?(?=\n\s*\*\s*\*\s*\*|\Z)",
    "", text, flags=re.S
)

blocks = re.split(r"(?m)^\s*\*\s*\*\s*\*\s*$", text)[1:]
letters = []

for block in blocks:
    block = block.strip()
    if not block:
         continue

   
    block = re.sub(r"\[\d+\s*-\s*[^\]]*\]|\[\d+\]", "", block)

    if block.startswith("Телеграмма"):
        telegrams = re.findall(r"Енисейск\..*?(?=Енисейск\.|\Z)", block, re.S)
        for telegram in telegrams:
            letters.append("Телеграмма\n" + telegram.strip())
    else:
         letters.append(block)

print("основных писем, открыток и телеграм:", len(letters))
print("записок внутри воспоминаний:", len(notes))

основных писем, открыток и телеграм: 76
записок внутри воспоминаний: 2


## 3. Определение автора

In [12]:
def find_author(letter):
    if re.search(r"(?:^|[.!?]\s+)Лина\.?\s*$", letter, re.M):
        return "Степанова", "подпись Лина"


    
    header = "\n".join(line for line in letter.splitlines()[:20] if len(line) < 120)
    if re.search(r"Степановой", header):
        return "Эрдман", "адресовано Степановой"
    if re.search(r"Эрдману", header):
        return "Степанова", "адресовано Эрдману"

    if re.search(r"Николай\.|Целую Коля", letter):
        return "Эрдман", "подпись Николай / Коля"
    if re.search(r"Линуш|\bЛина\b|Худыра|Пинчик", letter):
        return "Эрдман", "обращение к Степановой"
    if re.search(r"\bКоля\b|Николай|обожаемый мой", letter):
        return "Степанова", "обращение к Эрдману"
    if re.search(r"^Телеграмма\s+Енисейск", letter):
        return "Эрдман", "телеграмма из Енисейска; проверено по контексту"

    return None, "нужна проверка"

## 4. Определение разных вариантов дат

In [13]:
months = {
    "янв": 1, "фев": 2, "мар": 3, "апр": 4, "май": 5, "мая": 5,
    "июн": 6, "июл": 7, "авг": 8, "сен": 9, "окт": 10, "ноя": 11, "дек": 12
}
roman_months = {
    "I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6,
    "VII": 7, "VIII": 8, "IX": 9, "X": 10, "XI": 11, "XII": 12
}

def read_date(line):
    line = line.strip().strip("[]() ")
    line = re.sub(r"\s+г(?:ода|од|\.)?\s*$", "", line, flags=re.I).strip()

    match = re.fullmatch(r"(\d{1,2})[./-](\d{1,2}|[IVX]+)[./-](\d{4}|\d{2})", line, re.I)
    if match:
        day, month, year = match.groups()
        month = int(month) if month.isdigit() else roman_months.get(month.upper())
        year = int(year)
        if year < 100:
            year += 1900
        return year, month, int(day)

    match = re.fullmatch(r"(?:(\d{1,2})\s+)?([А-Яа-яЁё]+)\.?(?:\s+(\d{4}|\d{2}))?", line)
    if match:
        day, word, year = match.groups()
        month = months.get(word.lower()[:3])
        if month:
            year = int(year) if year else None
            if year is not None and year < 100:
                year += 1900
            return year, month, int(day) if day else None

    if re.fullmatch(r"19\d{2}", line):
        return int(line), None, None
    return None

def find_date(letter):
    for line in letter.splitlines():
        line = line.strip()
        candidate = re.sub(r"^Николай\.\s*", "", line)
        parts = read_date(candidate)
        if parts:
            return parts, candidate
    return (None, None, None), ""

rows = []
for letter in letters:
    author, author_rule = find_author(letter)
    parts, raw_date = find_date(letter)
    year, month, day = parts
    rows.append({
        "letter": letter, "author": author, "author_rule": author_rule,
        "year": year, "month": month, "day": day,
        "date_raw": raw_date, "year_inferred": False
    })

## 5. Работа с данными: восстанавливаю пропущенный год и сохраняю даты

In [14]:
# ту брал исходные даты с указанным годом
anchors = [(i, row) for i, row in enumerate(rows) if row["year"] is not None]

for i, row in enumerate(rows):
    if row["year"] is not None or row["month"] is None:
        continue

    before = [item for item in anchors if item[0] < i]
    after = [item for item in anchors if item[0] > i]
    if not before or not after:
        continue

    left = before[-1][1]
    right = after[0][1]
    possible_years = []
    for year in range(left["year"], right["year"] + 1):
        
        first = (year, row["month"], row["day"] or 1)
        last = (year, row["month"], row["day"] or 31)
        left_date = (left["year"], left["month"] or 1, left["day"] or 1)
        right_date = (right["year"], right["month"] or 12, right["day"] or 31)
        if last >= left_date and first <= right_date:
            possible_years.append(year)

    if len(possible_years) == 1:
        row["year"] = possible_years[0]
        row["year_inferred"] = True

# добавил записки после основной переписки, чтобы они не сбивали хронологию
for note in notes:
    rows.append({
        "letter": note.strip(), "author": "Эрдман",
        "author_rule": "заголовок Записки Н. Эрдмана",
        "year": 1931, "month": None, "day": None,
        "date_raw": "1931 (год указан перед записками)", "year_inferred": False
    })

def format_date(year, month, day):
    if year is None:
        return None
    if month is None:
        return str(year)
    if not 1 <= month <= 12:
        return None
    if day is None:
        return f"{year:04d}-{month:02d}"

    result = f"{year:04d}-{month:02d}-{day:02d}"
    
    # Проверил что такая календарная дата существует
    checked = pd.to_datetime(result, format="%Y-%m-%d", errors="coerce")
    return None if pd.isna(checked) else result

for row in rows:
    row["date"] = format_date(row["year"], row["month"], row["day"])

## 6. Очистка данных: чищу письма и считаю кол-во слов

In [16]:
def clean_letter(letter):
    letter = re.sub(r"\[\d+\s*-\s*[^\]]*\]|\[\d+\]", "", letter)
    lines = [line.strip() for line in letter.splitlines() if line.strip()]
    result = []
    header = True

    for line in lines:
        if read_date(line):
            continue
        if line.startswith("Николай.") and read_date(line[len("Николай."):]):
            line = "Николай."

        address = re.search(
            r"^(Москва|Баку|Батум|Чаква|Енисейск|Томск|Большая Мурта|Деревня Большая Мурта|"
            r"Оперный театр|проезд|Художественный|Ангелине|Ольге Алексеевне|"
            r"А\.\s*О\.\s*Степановой|Начальнику Строительства|Заказное|Телеграмма)", line
        )
        if header and len(line) < 120 and address:
            continue

        header = False
        result.append(line)

    return re.sub(r"\s+", " ", " ".join(result)).strip()

morph = pymorphy3.MorphAnalyzer()
dataset = []
date_checks = []

for number, row in enumerate(rows, start=1):
    clean_text = clean_letter(row["letter"])
    words = []
    for token in tokenize(clean_text.lower()):
        
        for word in re.findall(r"[а-яёa-z]+", token.text):
            words.append(word)

    lemmas = [morph.parse(word)[0].normal_form for word in words]
    dataset.append({
        "date": row["date"], "author": row["author"], "text": clean_text,
        "lemmas": " ".join(lemmas), "word_count": len(words)
    })
    date_checks.append({
        "letter_id": number, "date_raw": row["date_raw"], "date": row["date"],
        "year_inferred": row["year_inferred"], "author": row["author"],
        "author_rule": row["author_rule"]
    })

df = pd.DataFrame(dataset)
df["date"] = df["date"].astype("string")
df_dates = pd.DataFrame(date_checks)
print("общее кол-во писем и записок:", len(df))

общее кол-во писем и записок: 78


## 7. Финальный чек перед сохранением

In [17]:
df.info()
print("\nАвторы:")
print(df["author"].value_counts(dropna=False))

print("\nбез даты:", df["date"].isna().sum())
print("год восстановлен по контексту:", df_dates["year_inferred"].sum())
print("без автора:", df["author"].isna().sum())
print("пустых текстов:", (df["text"].str.len() == 0).sum())
print("остались переносы:", df["text"].str.contains(r"[\r\n]").any())
print("остались сноски:", df["text"].str.contains(r"\[\d+\]").any())

if df["author"].isna().any():
    raise ValueError("остались письма без автора: проверь df_dates")

df[["date", "author", "word_count"]].head(10)

<class 'pandas.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        62 non-null     string
 1   author      78 non-null     str   
 2   text        78 non-null     str   
 3   lemmas      78 non-null     str   
 4   word_count  78 non-null     int64 
dtypes: int64(1), str(3), string(1)
memory usage: 3.2 KB

Авторы:
author
Эрдман       47
Степанова    31
Name: count, dtype: int64

без даты: 16
год восстановлен по контексту: 31
без автора: 0
пустых текстов: 0
остались переносы: False
остались сноски: False


,date,author,word_count
0,<NA>,Эрдман,208
1,<NA>,Эрдман,500
2,1933-11-03,Эрдман,111
3,1933-11-04,Степанова,122
4,1933-11-05,Степанова,28
5,1933-11-06,Эрдман,185
6,1933-11-06,Степанова,131
7,1933-11-16,Степанова,246
8,1933-11-20,Эрдман,483
9,1933-11-22,Эрдман,363


## 8. Создание датасетп и отдельной таблицы проверки

In [22]:
df.to_csv("letters.csv", index=False, encoding="utf-8-sig")
df_dates.to_csv("dates_check.csv", index=False, encoding="utf-8-sig")


saved_df = pd.read_csv("letters.csv", dtype={"date": "string"}, encoding="utf-8-sig")
print("строк в сохранённом CSV:", len(saved_df))
print("файлы сохранены в:", os.getcwd())

строк в сохранённом CSV: 78
файлы сохранены в: c:\Users\Admin\Desktop\Данные для Python


## 9. Радуюсь жизни, прочитав басню Эрдмана